# Dataset Overview — Phase 3 Data Discovery

- **Project:** Rental Market Intelligence Platform
- **Source:** NYC Airbnb (Inside Airbnb–style export)
- **Snapshot date:** 01 August 2025 (scrape timestamps `2025-08-02` → `2025-08-03`)
- **Prepared:** 2026-07-14
- **Scope:** Discovery & profiling only. No cleaning, transformation, or modelling performed.

All figures below are measured directly from the raw files with pandas 3.0.3 / Python 3.13.7. Where a value could not be verified from the data, that is stated explicitly.

---

## 1. Files at a glance

| Dataset | Rows (data) | Columns | File size on disk | In-memory (deep) | Grain |
|---|---:|---:|---:|---:|---|
| `listings.csv` | 36,403 | 79 | 76,996,955 B (≈ 73.4 MB) | ≈ 142.7 MB | One row per listing |
| `calendar.csv` | 13,287,103 | 7 | 474,237,645 B (≈ 452.3 MB) | ≈ 1.84 GB | One row per listing per calendar date |
| `reviews.csv` | 977,459 | 6 | 306,884,115 B (≈ 292.7 MB) | ≈ 491.4 MB | One row per review event |

> **Note on row counts:** `listings.csv` contains embedded newlines inside free-text fields (`description`, `host_about`, `amenities`, etc.). A naive line count (`wc -l`) reports 77,866 lines, but the true parsed record count is **36,403**. Always parse with a proper CSV reader, not line counting.

---

## 2. `listings.csv`

- **Rows:** 36,403
- **Columns:** 79
- **File size:** 76,996,955 bytes (≈ 73.4 MB)
- **In-memory footprint (deep):** ≈ 142.7 MB
- **Grain:** One row = one Airbnb listing as observed in the 01-Aug-2025 scrape. `id` is unique across all rows (0 duplicates).
- **Purpose:** The **master / dimension-like table** describing each property and its host. Holds descriptive attributes (location, room/property type, capacity, amenities), host attributes (tenure, response behaviour, superhost status), pre-aggregated review scores, availability windows, and a single current listed `price`. This is the anchor table both other datasets reference.

---

## 3. `calendar.csv`

- **Rows:** 13,287,103
- **Columns:** 7
- **File size:** 474,237,645 bytes (≈ 452.3 MB)
- **In-memory footprint (deep):** ≈ 1.84 GB
- **Grain:** One row = the availability state of one listing on one calendar date. Coverage is exactly **365 dates per listing** (36,403 listings × 365 = 13,287,095; the file has 13,287,103, i.e. a handful of listings carry a small number of extra dated rows).
- **Date range:** `2025-08-02` → `2026-08-02` (a forward-looking 12-month booking calendar).
- **Purpose:** The **daily availability fact table**. It records, for the year ahead, whether each listing is available (`t`) or not (`f`) on each night, plus the minimum/maximum-nights rule in force for that date.
- **Critical limitation:** `price` and `adjusted_price` are **100% NULL** in this snapshot (see [data_quality_report.md](data_quality_report.md)). Calendar therefore provides *availability* but **not** per-night pricing.

---

## 4. `reviews.csv`

- **Rows:** 977,459
- **Columns:** 6
- **File size:** 306,884,115 bytes (≈ 292.7 MB)
- **In-memory footprint (deep):** ≈ 491.4 MB
- **Grain:** One row = one guest review left on a listing. `id` (review id) is unique (0 duplicates).
- **Date range:** `2009-05-25` → `2025-08-02`.
- **Distinct listings reviewed:** 25,093 (68.9% of all listings have ≥1 review).
- **Distinct reviewers:** 861,113.
- **Purpose:** The **review event fact table**. Because Airbnb only allows a review after a completed stay, each review is a proxy signal for a *booked & completed reservation*, which is the basis for demand / occupancy estimation.

---

## 5. Relationship summary (high level)

```
listings (1) ──< calendar (many)     via listing_id = listings.id   [exact 1:1 coverage of listing set]
listings (1) ──< reviews  (many)     via listing_id = listings.id   [subset: only reviewed listings]
```

- Every `calendar.listing_id` and every `reviews.listing_id` resolves to an existing `listings.id` — **0 orphan foreign keys** in either file.
- `calendar` covers **all** 36,403 listings; `reviews` covers the **25,093** listings that have received at least one review.

Full detail in [relationship_analysis.md](relationship_analysis.md).


# Schema Analysis — Phase 3 Data Discovery

Data types below are the **types as observed when read raw** (pandas inference), not proposed target types. Where the observed type is wrong for the business meaning (e.g. price read as string), that is flagged and carried into [data_quality_report.md](data_quality_report.md).

**Category legend**
- **Key** — primary identifier of the row
- **FK** — foreign key referencing another dataset
- **Dim** — dimension attribute (descriptive, used to slice/filter/group)
- **Fact** — numeric measure / metric
- **Meta** — operational/technical metadata (scrape bookkeeping, URLs, pre-computed derived fields)

---

## 1. `listings.csv` (79 columns)

| # | Column | Observed type | Description | Business meaning | Category |
|---:|---|---|---|---|---|
| 1 | id | int64 | Airbnb listing id | Primary key of a listing | **Key** |
| 2 | listing_url | str | Web URL of listing | Deep-link; derivable from id | Meta |
| 3 | scrape_id | int64 | Batch id of scrape (single value `20250801203054`) | Load/batch identifier | Meta |
| 4 | last_scraped | str (date) | Date row was scraped (2025-08-02/03) | Snapshot freshness | Meta |
| 5 | source | str | `city scrape` / `previous scrape` | How the record entered this snapshot | Meta |
| 6 | name | str | Listing title | Marketing headline | Dim |
| 7 | description | str | Long free-text description | Listing detail | Dim |
| 8 | neighborhood_overview | str | Host's blurb about the area | Area description | Dim |
| 9 | picture_url | str | Cover photo URL | Media asset | Meta |
| 10 | host_id | int64 | Airbnb host id | FK to host (host is not a separate file, but this groups listings) | **FK** (host) |
| 11 | host_url | str | Host profile URL | Media/link | Meta |
| 12 | host_name | str | Host display name | Host attribute | Dim |
| 13 | host_since | str (date) | Date host joined | Host tenure | Dim |
| 14 | host_location | str | Host's stated location | Host attribute | Dim |
| 15 | host_about | str | Host bio | Host attribute | Dim |
| 16 | host_response_time | str | Bucketed response speed | Host service quality | Dim |
| 17 | host_response_rate | str (`NN%`) | % of enquiries answered | Host service metric (stored as text) | Fact* |
| 18 | host_acceptance_rate | str (`NN%`) | % of requests accepted | Host service metric (stored as text) | Fact* |
| 19 | host_is_superhost | str (`t`/`f`) | Superhost flag | Quality badge (boolean-as-text) | Dim |
| 20 | host_thumbnail_url | str | Host thumbnail | Media | Meta |
| 21 | host_picture_url | str | Host photo | Media | Meta |
| 22 | host_neighbourhood | str | Host's neighbourhood | Host attribute | Dim |
| 23 | host_listings_count | float64 | Host's listing count (Airbnb-reported) | Host portfolio size | Fact |
| 24 | host_total_listings_count | float64 | Host's total listings incl. other types | Host portfolio size | Fact |
| 25 | host_verifications | str (list-like) | Verification methods | Host trust attribute | Dim |
| 26 | host_has_profile_pic | str (`t`/`f`) | Has profile pic | Boolean-as-text | Dim |
| 27 | host_identity_verified | str (`t`/`f`) | Identity verified | Boolean-as-text | Dim |
| 28 | neighbourhood | str | Raw neighbourhood text | Low quality (see below) | Dim |
| 29 | neighbourhood_cleansed | str | Standardised neighbourhood (223 distinct) | Reliable area dimension | Dim |
| 30 | neighbourhood_group_cleansed | str | Borough (5 distinct) | Reliable borough dimension | Dim |
| 31 | latitude | float64 | Latitude | Geo point | Dim |
| 32 | longitude | float64 | Longitude | Geo point | Dim |
| 33 | property_type | str | Detailed property type (74 distinct) | Product taxonomy | Dim |
| 34 | room_type | str | Room type (4 distinct) | Core product segment | Dim |
| 35 | accommodates | int64 | Max guests | Capacity | Fact |
| 36 | bathrooms | float64 | Bathroom count (numeric) | Capacity | Fact |
| 37 | bathrooms_text | str | Bathroom description | Capacity (text form) | Dim |
| 38 | bedrooms | float64 | Bedroom count | Capacity | Fact |
| 39 | beds | float64 | Bed count | Capacity | Fact |
| 40 | amenities | str (list-like) | JSON-ish array of amenities | Feature set | Dim |
| 41 | price | **str** (`$NNN.NN`) | Current nightly listed price | **Core money metric, stored as string** | Fact* |
| 42 | minimum_nights | int64 | Min stay | Booking rule | Fact |
| 43 | maximum_nights | int64 | Max stay | Booking rule | Fact |
| 44 | minimum_minimum_nights | float64 | Min of min-nights over calendar | Derived rule stat | Meta |
| 45 | maximum_minimum_nights | float64 | Max of min-nights over calendar | Derived rule stat | Meta |
| 46 | minimum_maximum_nights | float64 | Min of max-nights over calendar | Derived rule stat | Meta |
| 47 | maximum_maximum_nights | float64 | Max of max-nights over calendar | Derived rule stat | Meta |
| 48 | minimum_nights_avg_ntm | float64 | Avg min-nights next-twelve-months | Derived rule stat | Meta |
| 49 | maximum_nights_avg_ntm | float64 | Avg max-nights next-twelve-months | Derived rule stat | Meta |
| 50 | calendar_updated | float64 | Always NULL | Deprecated field | Meta |
| 51 | has_availability | str (`t`) | Availability flag | Only `t`/NULL present | Dim |
| 52 | availability_30 | int64 | Nights available next 30d | Availability metric | Fact |
| 53 | availability_60 | int64 | Nights available next 60d | Availability metric | Fact |
| 54 | availability_90 | int64 | Nights available next 90d | Availability metric | Fact |
| 55 | availability_365 | int64 | Nights available next 365d | Availability metric | Fact |
| 56 | calendar_last_scraped | str (date) | Calendar scrape date | Freshness | Meta |
| 57 | number_of_reviews | int64 | Total reviews all-time | Demand proxy | Fact |
| 58 | number_of_reviews_ltm | int64 | Reviews last 12 months | Recent demand proxy | Fact |
| 59 | number_of_reviews_l30d | int64 | Reviews last 30 days | Very recent demand | Fact |
| 60 | availability_eoy | int64 | Nights available to end of year | Availability metric | Fact |
| 61 | number_of_reviews_ly | int64 | Reviews last year | Demand proxy | Fact |
| 62 | estimated_occupancy_l365d | int64 | Airbnb/InsideAirbnb occupancy est. (nights) | **Pre-computed occupancy** | Fact |
| 63 | estimated_revenue_l365d | float64 | Estimated revenue last 365d | **Pre-computed revenue** | Fact |
| 64 | first_review | str (date) | Date of first review | Listing maturity | Dim |
| 65 | last_review | str (date) | Date of last review | Recency of activity | Dim |
| 66 | review_scores_rating | float64 | Overall rating (0–5) | Quality metric | Fact |
| 67 | review_scores_accuracy | float64 | Accuracy sub-score | Quality metric | Fact |
| 68 | review_scores_cleanliness | float64 | Cleanliness sub-score | Quality metric | Fact |
| 69 | review_scores_checkin | float64 | Check-in sub-score | Quality metric | Fact |
| 70 | review_scores_communication | float64 | Communication sub-score | Quality metric | Fact |
| 71 | review_scores_location | float64 | Location sub-score | Quality metric | Fact |
| 72 | review_scores_value | float64 | Value sub-score | Quality metric | Fact |
| 73 | license | str | STR registration / `Exempt` | Regulatory compliance | Dim |
| 74 | instant_bookable | str (`t`/`f`) | Instant-book flag | Boolean-as-text | Dim |
| 75 | calculated_host_listings_count | int64 | Host listings in this dataset | Host portfolio (computed) | Fact |
| 76 | calculated_host_listings_count_entire_homes | int64 | ‑ entire homes | Host portfolio | Fact |
| 77 | calculated_host_listings_count_private_rooms | int64 | ‑ private rooms | Host portfolio | Fact |
| 78 | calculated_host_listings_count_shared_rooms | int64 | ‑ shared rooms | Host portfolio | Fact |
| 79 | reviews_per_month | float64 | Avg reviews/month | Demand-rate proxy | Fact |

\* **Fact\*** = business measure that is currently stored in a non-numeric form (string with `$` or `%`) and must be cast before use.

---

## 2. `calendar.csv` (7 columns)

| # | Column | Observed type | Description | Business meaning | Category |
|---:|---|---|---|---|---|
| 1 | listing_id | int64 | Listing id | References `listings.id` | **FK / part of Key** |
| 2 | date | str (date) | Calendar night (2025-08-02 → 2026-08-02) | The night being described | **Key (composite with listing_id)** |
| 3 | available | str (`t`/`f`) | Available that night? | Availability state (boolean-as-text) | Fact/Dim |
| 4 | price | float64 | **100% NULL** | Intended per-night price — unusable here | Fact (empty) |
| 5 | adjusted_price | float64 | **100% NULL** | Intended adjusted price — unusable here | Fact (empty) |
| 6 | minimum_nights | int64 | Min stay for this date | Booking rule for the night | Fact |
| 7 | maximum_nights | int64 | Max stay for this date | Booking rule for the night | Fact |

**Composite primary key:** (`listing_id`, `date`).

---

## 3. `reviews.csv` (6 columns)

| # | Column | Observed type | Description | Business meaning | Category |
|---:|---|---|---|---|---|
| 1 | listing_id | int64 | Listing reviewed | References `listings.id` | **FK** |
| 2 | id | int64 | Review id | Primary key of a review | **Key** |
| 3 | date | str (date) | Date review posted | Timeline of demand | Dim/Fact |
| 4 | reviewer_id | int64 | Guest id | Reviewer identity | FK (guest) |
| 5 | reviewer_name | str | Guest first name | Reviewer attribute (4 nulls) | Dim |
| 6 | comments | str | Free-text review body | Sentiment/NLP source (260 nulls/empty) | Dim |

**Primary key:** `id`.


# Data Profiling — Phase 3 Data Discovery

All numbers are measured from the raw files. Percentages are of total data rows for that dataset.

---

## 1. `listings.csv` (36,403 rows × 79 cols)

- **Duplicate rows:** 0
- **Duplicate `id`:** 0 → `id` is a valid primary key
- **In-memory (deep):** ≈ 142.7 MB

### 1.1 Missing values — high-null columns

| Column | Nulls | % Null |
|---|---:|---:|
| calendar_updated | 36,403 | 100.00 |
| license | 30,937 | 84.98 |
| neighbourhood | 17,335 | 47.62 |
| neighborhood_overview | 17,336 | 47.62 |
| host_about | 15,439 | 42.41 |
| price | 15,124 | 41.55 |
| estimated_revenue_l365d | 15,124 | 41.55 |
| beds | 14,908 | 40.95 |
| bathrooms | 14,857 | 40.81 |
| host_acceptance_rate | 14,559 | 39.99 |
| host_response_rate | 14,531 | 39.92 |
| host_response_time | 14,531 | 39.92 |
| review_scores_* (7 cols) | ~11,310–11,348 | ~31.1 |
| first_review / last_review / reviews_per_month | 11,310 | 31.07 |
| host_location | 7,575 | 20.81 |
| host_neighbourhood | 7,392 | 20.31 |
| bedrooms | 5,920 | 16.26 |
| has_availability | 5,657 | 15.54 |

Columns with **0 nulls** include: `id`, `listing_url`, `scrape_id`, `last_scraped`, `source`, `host_id`, `neighbourhood_cleansed`, `neighbourhood_group_cleansed`, `latitude`, `longitude`, `property_type`, `room_type`, `accommodates`, `amenities`, `minimum_nights`, `maximum_nights`, all `availability_*`, all `number_of_reviews*`, `estimated_occupancy_l365d`, `instant_bookable`, and the `calculated_host_listings_count*` family.

> **Notable coupling:** `price` and `estimated_revenue_l365d` share the *same* 15,124 null rows — revenue is only computed where a price exists.
> The 7 `review_scores_*`, `first_review`, `last_review`, and `reviews_per_month` share ~11,310 nulls — these are exactly the listings with **no reviews** (36,403 − 25,093 reviewed = 11,310).

### 1.2 Cardinality (selected)

| Column | Distinct | Note |
|---|---:|---|
| id | 36,403 | unique key |
| host_id | 21,643 | many listings per host |
| scrape_id | 1 | constant |
| source | 2 | city scrape / previous scrape |
| neighbourhood_group_cleansed | 5 | boroughs |
| neighbourhood_cleansed | 223 | neighbourhoods |
| property_type | 74 | detailed taxonomy |
| room_type | 4 | core segment |
| neighbourhood (raw) | 1 | effectively useless (only "Neighborhood highlights") |
| has_availability | 1 | only `t` (rest null) |
| license | 1,978 | mostly null / `Exempt` |

### 1.3 Numeric statistics (selected)

| Column | mean | std | min | 25% | 50% | 75% | max |
|---|---:|---:|---:|---:|---:|---:|---:|
| accommodates | 2.74 | 1.87 | 1 | 2 | 2 | 4 | 16 |
| bedrooms | 1.39 | 0.94 | 0 | 1 | 1 | 2 | 16 |
| beds | 1.63 | 1.20 | 0 | 1 | 1 | 2 | 40 |
| bathrooms | 1.19 | 0.56 | 0 | 1 | 1 | 1 | 15.5 |
| minimum_nights | 28.6 | 29.3 | 1 | 30 | 30 | 30 | 1,124 |
| maximum_nights | 60,108 | 1.13e7 | 1 | 130 | 365 | 1,125 | 2,147,483,647 |
| availability_365 | 161.7 | 147.3 | 0 | 0 | 153 | 318 | 365 |
| number_of_reviews | 26.9 | 68.4 | 0 | 0 | 3 | 22 | 3,518 |
| estimated_occupancy_l365d | 47.2 | 85.0 | 0 | 0 | 0 | 60 | 255 |
| estimated_revenue_l365d | 14,782 | 97,335 | 0 | 0 | 0 | 18,000 | 12,763,260 |
| review_scores_rating | 4.73 | 0.45 | 0 | 4.65 | 4.86 | 5.0 | 5.0 |
| reviews_per_month | 0.82 | 1.88 | 0.01 | 0.08 | 0.25 | 0.92 | 123.87 |

**`price` (parsed from `$` string, non-null only, n=21,279):**

| mean | std | min | 25% | 50% | 75% | max |
|---:|---:|---:|---:|---:|---:|---:|
| 447.87 | 3,174.21 | 3 | 90 | 150 | 257 | 50,052 |

`price == 0`: 0 rows. (See outliers in [data_quality_report.md](data_quality_report.md).)

### 1.4 Categorical distributions (selected)

**room_type**

| value | count |
|---|---:|
| Entire home/apt | 19,328 |
| Private room | 16,469 |
| Hotel room | 378 |
| Shared room | 228 |

**neighbourhood_group_cleansed (borough)**

| borough | count |
|---|---:|
| Manhattan | 16,225 |
| Brooklyn | 13,322 |
| Queens | 5,336 |
| Bronx | 1,155 |
| Staten Island | 365 |

**host_is_superhost**: f = 28,842 · t = 7,147 · NULL = 414
**instant_bookable**: f = 29,000 · t = 7,403
**host_identity_verified**: t = 31,642 · f = 4,748 · NULL = 13
**source**: city scrape = 21,552 · previous scrape = 14,851

### 1.5 Date ranges

| Column | Min | Max | Nulls |
|---|---|---|---:|
| last_scraped | 2025-08-02 | 2025-08-03 | 0 |
| host_since | 2008-08-11 | 2025-07-30 | 13 |
| first_review | 2009-05-25 | 2025-08-01 | 11,310 |
| last_review | 2011-05-12 | 2025-08-02 | 11,310 |
| calendar_last_scraped | 2025-08-02 | 2025-08-03 | 0 |

### 1.6 Sample record (row 0, abridged)

```
id=2539 · name="Superfast Wi-Fi. Clean & quiet home by the park"
host_id=2787 · host_since=2008-09-07 · host_is_superhost=f
neighbourhood_group_cleansed=Brooklyn · neighbourhood_cleansed=Kensington
lat=40.64529 · lon=-73.97238 · room_type=Private room · property_type=Private room in condo
accommodates=2 · bedrooms=1 · beds=1 · bathrooms_text="1 shared bath"
price=$260.00 · minimum_nights=30 · maximum_nights=730 · availability_365=365
number_of_reviews=9 · review_scores_rating=4.89 · estimated_occupancy_l365d=0
```

---

## 2. `calendar.csv` (13,287,103 rows × 7 cols)

- **In-memory (deep):** ≈ 1.84 GB
- **Distinct listing_id:** 36,403 (== full listings set)
- **Rows per listing:** ≈ 365 (forward 12-month calendar)

### 2.1 Missing values

| Column | Nulls | % Null |
|---|---:|---:|
| listing_id | 0 | 0.00 |
| date | 0 | 0.00 |
| available | 0 | 0.00 |
| **price** | 13,287,103 | **100.00** |
| **adjusted_price** | 13,287,103 | **100.00** |
| minimum_nights | 0 | 0.00 |
| maximum_nights | 0 | 0.00 |

### 2.2 Distributions & ranges

- **available:** `f` = 7,399,750 (55.7%) · `t` = 5,887,353 (44.3%)
- **date range:** 2025-08-02 → 2026-08-02
- **minimum_nights:** min 1 · max 1,124
- **price / adjusted_price:** no parseable values at all (0 of 13.29M rows parse to a number)

### 2.3 Sample records

| listing_id | date | available | price | adjusted_price | minimum_nights | maximum_nights |
|---|---|---|---|---|---|---|
| 2539 | 2025-08-03 | t | (null) | (null) | 30 | 730 |
| 2539 | 2025-08-04 | t | (null) | (null) | 30 | 730 |
| 2539 | 2025-08-05 | t | (null) | (null) | 30 | 730 |

---

## 3. `reviews.csv` (977,459 rows × 6 cols)

- **In-memory (deep):** ≈ 491.4 MB
- **Duplicate review `id`:** 0
- **Distinct listing_id:** 25,093
- **Distinct reviewer_id:** 861,113

### 3.1 Missing values

| Column | Nulls | % Null |
|---|---:|---:|
| listing_id | 0 | 0.00 |
| id | 0 | 0.00 |
| date | 0 | 0.00 |
| reviewer_id | 0 | 0.00 |
| reviewer_name | 4 | 0.0004 |
| comments | 260 | 0.027 |

(+ the 260 `comments` include blank/whitespace-only bodies counted as empty.)

### 3.2 Ranges

- **date range:** 2009-05-25 → 2025-08-02 (16+ years of history)
- **reviewers:** 861,113 distinct → most guests review once; small repeat-guest tail.

### 3.3 Sample records

| listing_id | id | date | reviewer_id | reviewer_name | comments (truncated) |
|---|---|---|---|---|---|
| 2539 | 55688172 | 2015-12-04 | 25160947 | Peter | "Great host " |
| 2539 | 97474898 | 2016-08-27 | 91513326 | Liz | "Nice room for the price. Great neighborhood…" |
| 2539 | 105340344 | 2016-10-01 | 90022459 | Евгений | "Very nice apt. New remodeled." |

> `comments` contains multilingual, emoji, and multi-line UTF-8 content — encoding must be handled as UTF-8 end-to-end.


# Data Quality Report — Phase 3 Data Discovery

This document **identifies** issues only. Nothing is fixed. Each issue lists evidence measured from the raw files and a severity to help prioritise Silver-layer work later.

**Severity:** 🔴 High (blocks a business question or corrupts a metric) · 🟠 Medium (needs cleaning, workaround exists) · 🟡 Low (cosmetic / minor).

---

## 1. `listings.csv`

### 1.1 Wrong data types / values-stored-as-text

| # | Issue | Column(s) | Evidence | Severity |
|---|---|---|---|---|
| L1 | **Price stored as string** with `$` and `.00` | `price` | Sample `"$260.00"`; dtype = str | 🔴 |
| L2 | **Percentages stored as string** with `%` | `host_response_rate`, `host_acceptance_rate` | e.g. `"100%"`, `"0%"` | 🟠 |
| L3 | **Booleans stored as text** `t`/`f` | `host_is_superhost`, `host_has_profile_pic`, `host_identity_verified`, `instant_bookable`, `has_availability` | value_counts show `t`/`f` | 🟠 |
| L4 | **List/JSON stored as string** | `amenities`, `host_verifications` | `["Blender", …]`, `['email','phone']` | 🟠 |

### 1.2 Missing values (high null %)

| # | Column | % Null | Note | Severity |
|---|---|---:|---|---|
| L5 | `calendar_updated` | 100.00 | Entirely empty / deprecated — drop candidate | 🟠 |
| L6 | `license` | 84.98 | Mostly missing; `Exempt` = 3,186; regulatory signal weak | 🟠 |
| L7 | `price` | 41.55 | 15,124 listings have **no price** at all | 🔴 |
| L8 | `estimated_revenue_l365d` | 41.55 | Same rows as missing price | 🟠 |
| L9 | `bathrooms` / `beds` | 40.8 / 41.0 | Capacity gaps; `bathrooms_text` has only 0.35% null → better source | 🟠 |
| L10 | `host_response_rate` / `_acceptance_rate` / `_response_time` | ~40 | Host-behaviour metrics sparse | 🟠 |
| L11 | 7× `review_scores_*`, `first_review`, `last_review`, `reviews_per_month` | ~31 | = the 11,310 listings with 0 reviews (structural, not an error) | 🟡 |
| L12 | `neighbourhood`, `neighborhood_overview`, `host_about` | ~42–48 | Free text; low analytical value | 🟡 |

### 1.3 Useless / degenerate columns

| # | Issue | Column | Evidence | Severity |
|---|---|---|---|---|
| L13 | Single-value column | `scrape_id` | 1 distinct value | 🟡 |
| L14 | Raw neighbourhood unusable | `neighbourhood` | 1 distinct value (`"Neighborhood highlights"`) after nulls; use `neighbourhood_cleansed` instead | 🟠 |
| L15 | `has_availability` only `t`/null | `has_availability` | 1 distinct non-null value | 🟡 |

### 1.4 Invalid values & outliers

| # | Issue | Column | Evidence | Severity |
|---|---|---|---|---|
| L16 | **Sentinel / absurd max_nights** | `maximum_nights`, `minimum_maximum_nights`, `maximum_maximum_nights`, `maximum_nights_avg_ntm` | max = 2,147,483,647 (= 2^31−1, integer sentinel) | 🟠 |
| L17 | **Extreme price outliers** | `price` (parsed) | max = 50,052 vs median 150; std 3,174 ≫ mean 448 | 🟠 |
| L18 | **Extreme bed count** | `beds` | max = 40 for accommodates≤16 → suspect | 🟠 |
| L19 | **Zero-value review scores** | `review_scores_*` | min = 0 on a 1–5 scale → likely placeholder, not a real 0 | 🟠 |
| L20 | **`bathrooms` = 0** | `bathrooms` | min = 0 (may be legitimate "0 shared baths" studios — verify) | 🟡 |
| L21 | **Very large `minimum_nights`** | `minimum_nights` | max = 1,124 nights (~3 years) — plausible but extreme | 🟡 |

### 1.5 Consistency

| # | Issue | Detail | Severity |
|---|---|---|---|
| L22 | `bathrooms` (numeric) vs `bathrooms_text` disagree on completeness | numeric 40.8% null, text 0.35% null — two representations of the same fact | 🟠 |
| L23 | `room_type` vs `property_type` redundancy | 74 property types roll up into 4 room types; keep both but define hierarchy | 🟡 |
| L24 | Duplicate rows | **0** duplicate rows and **0** duplicate `id` — no dedup needed here | ✅ |

---

## 2. `calendar.csv`

| # | Issue | Column(s) | Evidence | Severity |
|---|---|---|---|---|
| C1 | **`price` is 100% NULL** | `price` | 13,287,103 / 13,287,103 null | 🔴 |
| C2 | **`adjusted_price` is 100% NULL** | `adjusted_price` | 13,287,103 / 13,287,103 null | 🔴 |
| C3 | **Boolean stored as text** | `available` | values `t`/`f` | 🟠 |
| C4 | `date` stored as string | `date` | needs cast to DATE | 🟠 |
| C5 | Extreme `minimum_nights` | `minimum_nights` | max = 1,124 | 🟡 |
| C6 | Slight grain irregularity | whole file | 13,287,103 rows vs 36,403 × 365 = 13,287,095 → 8 extra dated rows; confirm no per-listing duplicate (`listing_id`,`date`) before use | 🟠 |

> **Consequence of C1/C2:** Calendar cannot contribute *any* pricing information in this snapshot. Any per-night or seasonal pricing analysis must fall back to the single `listings.price`, or the calendar price column must be re-sourced.

---

## 3. `reviews.csv`

| # | Issue | Column(s) | Evidence | Severity |
|---|---|---|---|---|
| R1 | Null / empty `comments` | `comments` | 260 rows null or whitespace-only | 🟡 |
| R2 | Null `reviewer_name` | `reviewer_name` | 4 rows | 🟡 |
| R3 | `date` stored as string | `date` | cast to DATE | 🟠 |
| R4 | Automated / non-guest comments | `comments` | Airbnb inserts canned text (e.g. cancellation notices) — needs filtering for sentiment work (present in Inside Airbnb data generally; flag for Silver) | 🟠 |
| R5 | Duplicate review `id` | `id` | **0** duplicates — clean | ✅ |
| R6 | Orphan foreign keys | `listing_id` | **0** — every review resolves to a listing | ✅ |

---

## 4. Cross-dataset issues

| # | Issue | Detail | Severity |
|---|---|---|---|
| X1 | **Pricing exists in only one place** | `listings.price` (41.6% null) is the *only* price signal; `calendar.price` is empty | 🔴 |
| X2 | Reviewed-listing gap | 11,310 listings (31%) have no reviews → any review-derived demand metric is undefined for them | 🟠 |
| X3 | Snapshot-only | Single 01-Aug-2025 snapshot; no history of `listings` over time → cannot see price/attribute change across scrapes | 🟠 |

---

## 5. Summary counts

| Dataset | Duplicate rows | Duplicate PK | 100%-null columns | Type-mismatch columns |
|---|---:|---:|---:|---:|
| listings | 0 | 0 | 1 (`calendar_updated`) | ≥ 8 (price, 2 rates, 5 booleans, 2 list cols) |
| calendar | (verify C6) | (verify C6) | 2 (`price`, `adjusted_price`) | 2 (`available`, `date`) |
| reviews | 0 | 0 | 0 | 1 (`date`) |


# Relationship Analysis — Phase 3 Data Discovery

All relationships below were **verified against the actual data**, not assumed from column names.

---

## 1. Primary keys

| Dataset | Primary key | Verified |
|---|---|---|
| `listings` | `id` | ✅ 36,403 rows, 36,403 distinct `id`, 0 duplicates |
| `calendar` | (`listing_id`, `date`) composite | ✅ grain is one row per listing per date (see note on 8 extra rows in [data_quality_report.md](data_quality_report.md) C6) |
| `reviews` | `id` | ✅ 977,459 rows, 0 duplicate review `id` |

## 2. Foreign keys

| From | Column | → To | Column | Integrity (measured) |
|---|---|---|---|---|
| `calendar` | `listing_id` | `listings` | `id` | ✅ 36,403 distinct FKs, **0 orphans**; set is **exactly equal** to the listings key set |
| `reviews` | `listing_id` | `listings` | `id` | ✅ 25,093 distinct FKs, **0 orphans**; strict **subset** of listings keys |
| `listings` | `host_id` | *(no host table)* | — | Implicit host grouping only; 21,643 distinct hosts. No standalone host dataset provided |
| `reviews` | `reviewer_id` | *(no reviewer table)* | — | 861,113 distinct reviewers; no reviewer dimension provided |

## 3. Join columns

| Join | Key(s) | Cardinality | Notes |
|---|---|---|---|
| listings ↔ calendar | `listings.id = calendar.listing_id` | 1 : ~365 | Every listing has a ~1-year forward calendar |
| listings ↔ reviews | `listings.id = reviews.listing_id` | 1 : 0..N | 31% of listings have 0 reviews |
| calendar ↔ reviews | (only via listings) | — | No direct key; must route through `listings` |

## 4. Relationship characteristics

- **listings → calendar** is a **complete (total) 1-to-many**: the calendar `listing_id` set equals the listings key set exactly. No listing lacks a calendar; no calendar row references a missing listing.
- **listings → reviews** is a **partial 1-to-many**: only reviewed listings appear. This matches the 11,310 listings with null `first_review`.
- `host_id` and `reviewer_id` point to **conceptual entities that have no dimension table** in this delivery. If a Host or Guest dimension is needed later, it must be **derived** from `listings` / `reviews` respectively.

## 5. ER diagram

```mermaid
erDiagram
    LISTINGS ||--o{ CALENDAR : "has daily availability"
    LISTINGS ||--o{ REVIEWS  : "receives"
    HOST ||--o{ LISTINGS     : "owns (implicit, no table)"
    GUEST ||--o{ REVIEWS     : "writes (implicit, no table)"

    LISTINGS {
        int64  id PK
        int64  host_id "FK-implicit"
        string neighbourhood_group_cleansed
        string neighbourhood_cleansed
        string room_type
        string property_type
        int    accommodates
        string price "stored as text $"
        float  review_scores_rating
        int    number_of_reviews
        int    availability_365
        int    estimated_occupancy_l365d
    }

    CALENDAR {
        int64  listing_id FK
        date   date PK
        string available "t/f"
        float  price "100% NULL"
        float  adjusted_price "100% NULL"
        int    minimum_nights
        int    maximum_nights
    }

    REVIEWS {
        int64  id PK
        int64  listing_id FK
        date   date
        int64  reviewer_id "FK-implicit"
        string reviewer_name
        string comments
    }

    HOST {
        int64 host_id PK "derive from listings"
    }
    GUEST {
        int64 reviewer_id PK "derive from reviews"
    }
```

## 6. Implications for modelling (not designed yet)

- `listings` is the natural **conformed dimension anchor** (listing + geography + host attributes).
- `calendar` is a **date-grained fact** (availability), and `reviews` is an **event-grained fact** (review/demand proxy), both joinable to `listings` on the listing key.
- A shared **Date dimension** could conform `calendar.date` and `reviews.date`.
- Host and Guest dimensions, if required, are **derivable but not supplied**.


# Business Understanding & Question Validation — Phase 3 Data Discovery

Part A explains *why* the important columns exist and whether they are likely Gold-layer material. Part B tests each target business question against what the data can actually support.

---

## Part A — Business meaning of key columns

| Column | Why it exists | Business questions it supports | Likely in Gold? |
|---|---|---|---|
| `listings.id` | Unique listing identity | All per-listing analysis; join key | ✅ Yes (dim key) |
| `neighbourhood_group_cleansed` | Standardised borough | Supply/price by borough | ✅ Yes (dim) |
| `neighbourhood_cleansed` | Standardised neighbourhood (223) | Fine-grained supply/price geography | ✅ Yes (dim) |
| `latitude` / `longitude` | Precise geolocation | Map viz, spatial clustering | ⚪ Maybe (geo) |
| `room_type` / `property_type` | Product taxonomy | Price/supply by product type | ✅ Yes (dim) |
| `accommodates`, `bedrooms`, `beds`, `bathrooms` | Capacity | Price normalisation (price per guest/bed) | ✅ Yes (dim/measure) |
| `price` | Current nightly rate | **Price questions** (core money metric) | ✅ Yes (fact) — after cast |
| `minimum_nights` / `maximum_nights` | Booking rules | Segment short- vs long-stay supply | ⚪ Maybe |
| `availability_30/60/90/365`, `availability_eoy` | Pre-computed availability windows | Supply/occupancy proxy | ✅ Yes (fact) |
| `estimated_occupancy_l365d` | Vendor occupancy estimate (nights) | **Occupancy** (ready-made) | ✅ Yes (fact) |
| `estimated_revenue_l365d` | Vendor revenue estimate | Revenue analysis | ✅ Yes (fact) — but 41.6% null |
| `number_of_reviews`, `_ltm`, `_l30d`, `_ly`, `reviews_per_month` | Review counts | **Demand proxy** over time windows | ✅ Yes (fact) |
| `review_scores_*` | Quality ratings | Quality vs price/demand | ✅ Yes (fact) |
| `host_is_superhost`, `host_since`, response/acceptance | Host quality/behaviour | Host-quality segmentation | ⚪ Maybe (dim) |
| `license` | STR regulatory status | Compliance analysis | ⚪ Maybe (sparse) |
| `calendar.date` + `available` | Daily availability state | **Time-based supply/occupancy** | ✅ Yes (fact) |
| `calendar.price` | Intended per-night price | Per-night/seasonal pricing | ❌ No — 100% null here |
| `reviews.date` | Review timeline | **Demand over time** (proxy) | ✅ Yes (fact) |
| `reviews.comments` | Review text | Sentiment/NLP (optional) | ⚪ Maybe (NLP) |

---

## Part B — Can we answer the target questions?

Verdict scale: ✅ Yes · 🟡 Partially / with assumptions · ❌ No (not from these datasets).

### Q1. Which neighbourhoods have the highest prices? — ✅ Yes (with caveats)
- **Data:** `listings.price` + `neighbourhood_cleansed` / `neighbourhood_group_cleansed`.
- **How:** cast price from `$` string to numeric, group by neighbourhood, compute median/mean.
- **Caveats / assumptions:**
  - `price` is **41.6% null** (15,124 listings) — the answer covers only the 21,279 priced listings; assume they are representative (unverified).
  - Extreme outliers (max $50,052, std $3,174) → **median** is safer than mean.
  - Price is a *listed* rate, not a *transacted* rate.

### Q2. Which neighbourhoods have the highest supply? — ✅ Yes
- **Data:** count of `listings.id` grouped by neighbourhood/borough (0 nulls on both geo columns).
- **How:** simple count; optionally weight by `availability_365` for *available* supply.
- **Caveats:** "supply" = listed supply in this snapshot; measured perfectly. Borough distribution already visible: Manhattan 16,225 · Brooklyn 13,322 · Queens 5,336 · Bronx 1,155 · Staten Island 365.

### Q3. How do prices change over time? — 🟡 Partially → effectively ❌ for true time series
- **What we *cannot* do:** build a real nightly/seasonal price time series. `calendar.price` and `adjusted_price` are **100% null**, and only **one** listings snapshot exists (no scrape history).
- **What we *can* do (assumption-heavy):** nothing reliable. There is no second time point and no dated price.
- **Verdict:** With only this snapshot, price-over-time **cannot be answered directly.** It becomes answerable only if (a) multiple snapshots are collected over time, or (b) `calendar.price` is re-sourced.

### Q4. Can occupancy be estimated? — 🟡 Yes, estimated (not measured)
- **Path A (ready-made):** `listings.estimated_occupancy_l365d` — a vendor-provided estimate in nights (0 nulls). Directly usable.
- **Path B (availability-based):** `1 − availability_365/365` gives a booked-share proxy from `calendar`/listings availability.
- **Path C (review-based / "San Francisco model"):** infer bookings from review frequency (`number_of_reviews_ltm`, `reviews_per_month`) × assumed review rate × avg stay length.
- **Assumptions/limits:** availability≠occupancy (a blocked night may be host-blocked, not booked); review-based models assume a fixed review probability. All three are **estimates**, explicitly not actual bookings.

### Q5. Can demand be estimated? — 🟡 Yes, proxied
- **Data:** `reviews` volume over time (`date`), and listings' `number_of_reviews_*` / `reviews_per_month`.
- **How:** reviews are a lower-bound proxy for completed stays (only a fraction of guests review). Aggregate reviews by month/neighbourhood as a demand index.
- **Assumptions/limits:** review-to-booking ratio is unknown and assumed constant; ignores browsing/enquiry demand and cancelled stays; 31% of listings have no reviews.

### Q6. Which questions cannot be answered directly?
| Question | Status | Blocker |
|---|---|---|
| True **price-over-time / seasonality** | ❌ | `calendar.price` 100% null + single snapshot |
| Actual **bookings / transacted revenue** | ❌ | No reservation/transaction data; only estimates & proxies |
| Actual **occupancy** (vs estimate) | ❌ | No booking records; availability & reviews are proxies |
| **Guest demographics / search demand** | ❌ | No guest profile or search-funnel data |
| **Host revenue actuals** | ❌ | Only `estimated_revenue_l365d` (modelled, 41.6% null) |
| Price **per night by date** | ❌ | Calendar price empty |

---

## Part C — Global assumptions & limitations

1. **Single snapshot (01-Aug-2025).** No historical dimension for listings → no change-over-time on listing attributes/prices.
2. **Listed ≠ transacted.** Prices are asking prices; occupancy/demand are estimates/proxies, never observed bookings.
3. **Calendar contributes availability only** — its price columns are unusable in this delivery.
4. **Coverage gaps:** price (41.6% null), reviews (31% of listings none), host-behaviour rates (~40% null) — subset analyses assume the covered subset is representative (not independently verified).
5. **Outliers present** in price and nights fields; use robust statistics (median/percentiles) until Silver-layer treatment.


# Data Engineering Observations — Phase 3 Data Discovery

Observations derived from profiling to inform later design. **No warehouse, star schema, or Bronze/Silver/Gold is designed here** — these are candidate ideas only.

---

## 1. Bronze Layer responsibilities (raw ingestion)

- Ingest all three CSVs **as-is**, no casting — preserve `price` as string, booleans as `t`/`f`, calendar `price` nulls, etc.
- **Force UTF-8** read/write end-to-end (reviews contain multilingual + emoji text; a naive cp1252 read fails).
- Use a **proper multiline-aware CSV parser** for `listings` (embedded newlines → line-count ≠ record-count; true count is 36,403).
- Capture ingestion metadata: source filename, load timestamp, `scrape_id` (`20250801203054`), `last_scraped`, row counts for reconciliation.
- Enforce **schema-on-read** column list (79 / 7 / 6) so drift in future snapshots is detected.
- Keep `calendar` partition-friendly (it is 452 MB / 13.3 M rows) — read in chunks or as Parquet.

## 2. Silver Layer cleaning tasks (candidates)

| Task | Columns | Action (later) |
|---|---|---|
| Cast price → numeric | `listings.price` | strip `$` `,`, cast decimal; keep null where absent |
| Cast percentages → numeric | `host_response_rate`, `host_acceptance_rate` | strip `%`, /100 |
| Cast booleans | `host_is_superhost`, `host_has_profile_pic`, `host_identity_verified`, `instant_bookable`, `has_availability`, `calendar.available` | `t/f` → boolean |
| Cast dates | `last_scraped`, `host_since`, `first_review`, `last_review`, `calendar.date`, `reviews.date` | → DATE |
| Parse arrays | `amenities`, `host_verifications` | explode / normalise |
| Handle sentinels | `maximum_nights` & family = 2^31−1 | flag/clip |
| Reconcile bathrooms | `bathrooms` vs `bathrooms_text` | derive numeric from text (text is 0.35% null vs 40.8%) |
| Drop/observe dead columns | `calendar_updated` (100% null), `neighbourhood` (1 val), `scrape_id` (constant) | quarantine |
| Null strategy | price, review_scores, host rates | keep null (do not impute silently); document |
| Calendar price | `calendar.price`, `adjusted_price` | mark unavailable this snapshot; do not fabricate |
| Dedup check | `calendar` (`listing_id`,`date`) | verify the 8 extra rows aren't dup keys |
| Comment filtering | `reviews.comments` | flag automated/canned + empty (260) for NLP |

## 3. Gold Layer candidate metrics

- **Price metrics:** median/mean nightly price by borough / neighbourhood / room_type; price per guest (`price/accommodates`), price per bedroom.
- **Supply metrics:** listing counts and *available* listing counts by geography/type; active vs inactive supply.
- **Availability/occupancy:** `1 − availability_365/365`; adopt `estimated_occupancy_l365d`; availability trend from calendar by month.
- **Demand proxies:** reviews per month, `number_of_reviews_ltm`, new-review velocity by neighbourhood/time.
- **Quality:** avg `review_scores_rating` and sub-scores; superhost share.
- **Revenue (estimate):** `estimated_revenue_l365d` roll-ups (flag as modelled, 41.6% null).

## 4. Candidate dimension tables (derivable — not designed yet)

- **Dim Listing** (from `listings`: id, geo, room/property type, capacity, rules, attributes).
- **Dim Host** (derive from `listings.host_id` + host_* columns; 21,643 hosts).
- **Dim Neighbourhood / Borough** (223 neighbourhoods → 5 boroughs; a clean geo hierarchy).
- **Dim Date** (conforms `calendar.date` 2025-08→2026-08 and `reviews.date` 2009→2025).
- **Dim Guest/Reviewer** (optional, from `reviews.reviewer_id`; 861,113).
- **Dim Property/Room Type** (taxonomy: 74 property → 4 room types).

## 5. Candidate fact tables (candidates)

- **Fact Availability (daily)** — grain (`listing_id`, `date`) from `calendar` (13.3 M rows): `available`, `minimum_nights`, `maximum_nights`. (No price this snapshot.)
- **Fact Review (event)** — grain `reviews.id` (977 K rows): listing, date, reviewer → demand index.
- **Fact Listing Snapshot** — grain (`listing_id`, `scrape_id`) from `listings`: price, availability windows, review scores, estimated occupancy/revenue. Snapshot-typed for future accumulation.

## 6. Incremental loading opportunities

- **`reviews`** is naturally **append-only** by `date` → incremental high-watermark load on `date` / `id` (immutable events).
- **`calendar`** is a **full forward snapshot** per scrape (rolling 12 months) → load as a dated partition set per `scrape_id`; new snapshots replace/append by scrape date rather than row-level upsert.
- **`listings`** → **snapshot per scrape** (SCD-style). This delivery has a single `scrape_id`; future scrapes enable Type-2 history and would finally unlock price-over-time (currently blocked).
- Use `last_scraped` / `scrape_id` / `reviews.date` as watermarks.

## 7. Candidate partition columns

| Dataset | Partition candidate | Rationale |
|---|---|---|
| calendar | `date` (month) | 13.3 M rows, all time-based scans; monthly partitions prune well |
| reviews | `date` (year or year-month) | 16-year span, append-only, time-range queries |
| listings | `scrape_id` / `last_scraped` | snapshot isolation; future multi-snapshot growth |
| (any) | `neighbourhood_group_cleansed` | secondary partition for borough-scoped analytics |

## 8. Candidate clustering / sort columns

| Dataset | Cluster/sort candidate | Rationale |
|---|---|---|
| calendar | `listing_id` (then `date`) | join key + range scans per listing |
| reviews | `listing_id` (then `date`) | per-listing review lookups |
| listings | `neighbourhood_cleansed`, `room_type` | common group-by/filter predicates |
| listings | `id` | primary-key point lookups & joins |

---

### Cross-cutting engineering notes
- **Volume:** calendar ≈ 1.84 GB in memory — prefer Parquet + chunked/columnar processing over full pandas loads.
- **Referential integrity is already clean** (0 orphan FKs both ways) → FK enforcement in Silver is validation, not repair.
- **Pricing is the key data risk:** the only usable price is `listings.price` (41.6% null); calendar price is empty. Any pricing product feature must design around this.


# Phase 3 — Data Discovery Summary

**Project:** Rental Market Intelligence Platform · **Snapshot:** 01 August 2025 · **Compiled:** 2026-07-14
**Scope:** Discovery & profiling only — no data was modified, cleaned, transformed, or modelled.

---

## 1. What we have

| Dataset | Rows | Cols | Size | Grain | Role |
|---|---:|---:|---:|---|---|
| listings.csv | 36,403 | 79 | ≈ 73.4 MB | 1 per listing | Master / dimension anchor |
| calendar.csv | 13,287,103 | 7 | ≈ 452.3 MB | 1 per listing per date | Daily availability fact |
| reviews.csv | 977,459 | 6 | ≈ 292.7 MB | 1 per review | Review / demand-proxy fact |

- Geography: 5 boroughs, 223 neighbourhoods. Manhattan (16,225) & Brooklyn (13,322) dominate supply.
- 21,643 distinct hosts; 861,113 distinct reviewers; review history 2009→2025; calendar covers 2025-08→2026-08.

## 2. Relationships (verified from data)

```mermaid
erDiagram
    LISTINGS ||--o{ CALENDAR : "listing_id (0 orphans, full coverage)"
    LISTINGS ||--o{ REVIEWS  : "listing_id (0 orphans, 25,093 of 36,403 listings)"
```

- **Primary keys:** `listings.id` (unique), `reviews.id` (unique), `calendar` = (`listing_id`,`date`).
- **Foreign keys:** both `calendar.listing_id` and `reviews.listing_id` resolve fully to `listings.id` — **0 orphans**.
- `host_id` / `reviewer_id` reference entities with **no supplied dimension table** (derivable later).

## 3. Top data-quality findings

| Sev | Finding |
|---|---|
| 🔴 | `calendar.price` & `adjusted_price` are **100% NULL** — no per-night pricing available. |
| 🔴 | `listings.price` is a **string** (`$260.00`) and **41.6% null** (only 21,279 of 36,403 priced). |
| 🟠 | Booleans stored as text `t/f` (6 columns across listings + calendar). |
| 🟠 | Percentages stored as text (`host_response_rate`, `host_acceptance_rate`, ~40% null). |
| 🟠 | Integer sentinel `2,147,483,647` in `maximum_nights` family; price/beds/min_nights outliers. |
| 🟠 | Dead/degenerate columns: `calendar_updated` (100% null), `neighbourhood` (1 value), `scrape_id` (constant). |
| 🟡 | 31% of listings have no reviews → review-based metrics undefined there (structural). |
| ✅ | **No duplicate rows or duplicate primary keys** in any dataset; **no orphan foreign keys**. |

## 4. Business questions — verdict

| Question | Verdict | Note |
|---|---|---|
| Highest-price neighbourhoods | ✅ Yes | via `listings.price` (cast); use median; 41.6% null caveat |
| Highest-supply neighbourhoods | ✅ Yes | listing counts by geography (0 nulls) |
| Prices over time | ❌ No | calendar price empty + single snapshot |
| Occupancy estimate | 🟡 Estimate | `estimated_occupancy_l365d`, availability, or review-based proxy |
| Demand estimate | 🟡 Proxy | review volume over time (assumes constant review rate) |
| Actual bookings / revenue | ❌ No | no transaction data; only vendor estimates |

## 5. Engineering signal (candidates, not designed)

- **Bronze:** raw as-is, UTF-8, multiline-aware parse, capture `scrape_id`/load metadata.
- **Silver:** cast price/%/booleans/dates, parse `amenities`, treat sentinels & outliers, reconcile `bathrooms`↔`bathrooms_text`, keep nulls explicit.
- **Gold candidates:** price (median by geo/type), supply counts, availability/occupancy, review-based demand, quality scores.
- **Dims:** Listing, Host, Neighbourhood/Borough, Date, (Guest, Property/Room type).
- **Facts:** Availability (daily), Review (event), Listing Snapshot.
- **Partitioning:** calendar/reviews by `date`; listings by `scrape_id`. **Cluster/sort:** by `listing_id`, geo, `room_type`.
- **Incremental:** reviews append-only by date; calendar snapshot-per-scrape; listings SCD-per-scrape.

## 6. Key limitations to carry forward

1. **Single snapshot** → no listing/price history (blocks true time-series).
2. **Pricing is the primary risk** → only `listings.price` usable, and it's 41.6% null.
3. **Occupancy & demand are estimates/proxies**, never observed bookings.
4. **No host/guest/transaction/search data** → those questions are out of scope with current inputs.

## 7. Document index

| File | Contents |
|---|---|
| [dataset_overview.md](dataset_overview.md) | Rows, cols, size, grain, purpose per dataset |
| [schema_analysis.md](schema_analysis.md) | Every column: type, description, meaning, category |
| [data_profiling.md](data_profiling.md) | Nulls, cardinality, stats, distributions, dates, samples |
| [data_quality_report.md](data_quality_report.md) | All issues catalogued by severity (no fixes) |
| [relationship_analysis.md](relationship_analysis.md) | PK/FK/joins + Mermaid ER diagram |
| [business_validation.md](business_validation.md) | Column business meaning + question feasibility |
| [engineering_observations.md](engineering_observations.md) | Bronze/Silver/Gold, dims/facts, partition/cluster, incremental |
| [phase3_summary.md](phase3_summary.md) | This summary |

---

### Recommended next phase
Proceed to **Silver-layer cleaning design** targeting the price/boolean/percentage casts and the calendar-price gap, and begin collecting **additional snapshots** so price-over-time becomes answerable. Warehouse/star-schema design remains explicitly out of scope until modelling phase.

> **Verification note:** Every figure in these documents was measured directly from the raw CSVs (pandas 3.0.3, Python 3.13.7). Where a fact could not be verified from the data (e.g. true occupancy, transacted price), it is labelled as an estimate, proxy, or explicitly "cannot be answered."
